In [ ]:
import os
import pandas as pd
from datetime import datetime
from lightningrod.utils import config
from dotenv import load_dotenv

load_dotenv()

from lightningrod import (
    LightningRod,
    BinaryAnswerType,
    GdeltSeedGenerator,
    ForwardLookingQuestionGenerator,
    QuestionPipeline,
    QuestionRenderer,
    WebSearchLabeler,
    NewsContextGenerator,
)

api_key = config.get_config_value("LIGHTNINGROD_API_KEY")
if not api_key:
    raise ValueError("LIGHTNINGROD_API_KEY is not set")

STAGING_BASE_URL = "https://lightningrod-api-staging-918054920018.us-central1.run.app/api/public/v1"
base_url = config.get_config_value("LIGHTNINGROD_BASE_URL", STAGING_BASE_URL)
if not base_url:
    raise ValueError("LIGHTNINGROD_BASE_URL is not set")

lr = LightningRod(api_key=api_key, base_url=base_url)

In [ ]:
instructions = """
Generate forward-looking binary forecasting questions with exactly one correct yes/no answer.

Each question must ask about a future event or outcome that is unresolved at the time of asking and will resolve within 3 months to a clear, publicly verifiable yes or no.

The outcome must be materially important, widely reported, and of real-world consequence (e.g., affecting economies, markets, elections, public policy, science, or competitive results).

Each question MUST:
- Be forward-looking and unresolved when asked
- Resolve within 3 months of the question date
- Have EXACTLY ONE binary answer: Yes or No
- Be fully self-contained (all entities, locations, dates included)
- Refer to a clearly defined event or threshold with an explicit resolution date or deadline
- Be resolvable via public web search using a single authoritative source
- Describe an outcome plausibly reported in a major news headline or official release
- Start with words like 'Will', 'Is', 'Does', 'Has', 'Can', 'Did', or similar

High-value domains including but not limited to economics, financial markets, elections, sports, science/environment, and public policy.

STRICTLY DO NOT include:
- Numeric or continuous outcomes
- Multiple-choice or categorical questions
- Questions that cannot fully resolve within 3 months
- Trivial, obscure, or low-impact events
- Vague language or ambiguous resolution criteria
- Outcomes dependent on unpublished, proprietary, or speculative data
- Questions with more than two possible outcomes
"""

good_examples = [
    # Economics / Central Banking
    "Question: Will the U.S. Federal Reserve cut interest rates at its Federal Open Market Committee meeting ending May 6, 2026?",

    # Financial Markets
    "Question: Will the S&P 500 index close above 5,000 on April 30, 2026?",

    # Elections / Politics
    "Question: Will the Liberal Party win a majority government in the Canadian federal election scheduled for April 28, 2026?",

    # Sports
    "Question: Will the UEFA Champions League final scheduled for May 30, 2026 go to extra time?",

    # Public Policy / Legislation
    "Question: Will the European Union formally adopt the Corporate Sustainability Due Diligence Directive before June 30, 2026?",

    # Science / Space
    "Question: Will NASA's Artemis II crewed lunar flyby mission launch before June 30, 2026?",

    # Technology / Corporate
    "Question: Will Apple announce a new iPhone model at its Worldwide Developers Conference in June 2026?",

    # Geopolitics / Diplomacy
    "Question: Will the United Nations Security Council vote to extend the UNIFIL peacekeeping mandate in Lebanon before its expiration on May 31, 2026?",

    # Environment / Climate
    "Question: Will NOAA declare the month of April 2026 the hottest April on record in its global temperature report?",

    # Labor / Employment
    "Question: Will the United Kingdom's unemployment rate fall below 4.0% in the February 2026 release from the Office for National Statistics?",

    # Commodities
    "Question: Will Brent crude oil futures close above $80 per barrel on June 30, 2026?",

    # Consumer Prices
    "Question: Will the U.S. Consumer Price Index year-over-year inflation rate exceed 3.0% for the month of March 2026 as reported by the Bureau of Labor Statistics?",
]

bad_examples = [
    "Question: What will the inflation rate be in March 2026?\n"
    "# BAD: Asks for a numeric value, not a binary yes/no answer.",

    "Question: Will inflation rise?\n"
    "# BAD: No time frame, no specific metric, no resolution date.",

    "Question: Which party will win the Canadian federal election?\n"
    "# BAD: Multiple-choice outcome, not binary yes/no.",

    "Question: Will the economy improve next year?\n"
    "# BAD: Vague metric, resolution period exceeds 3 months, no authoritative source.",

    "Question: Will LeBron James play in the next NBA game?\n"
    "# BAD: Trivial, timing is uncertain, not tied to a specific date.",

    "Question: Will the U.S. avoid a recession in 2026?\n"
    "# BAD: Resolution period exceeds 3 months; recession determination is lagging and ambiguous.",

    "Question: Will a major tech company have a data breach this quarter?\n"
    "# BAD: No specific company named, 'major' is subjective, 'data breach' lacks a clear threshold.",

    "Question: Will scientists discover a new species in the Amazon?\n"
    "# BAD: No time frame, no specific institution, discovery announcements are unpredictable and not tied to a deadline.",

    "Question: Will Tesla's stock go up tomorrow?\n"
    "# BAD: Trivial time horizon, not materially important as a single-day move, not the kind of event reported in a major headline.",
]

LRL_BINARY_ANSWER_FORMAT = "Think carefully in English about your answer and output your final prediction (a float between 0.0 and 1.0) between <answer></answer> tags. Example Outputs. These are just examples to illustrate the format and SHOULD NOT be considered baselines: <answer>0.75</answer>"

In [ ]:
answer_type = BinaryAnswerType(answer_format_instruction=LRL_BINARY_ANSWER_FORMAT)

pipeline = QuestionPipeline(
    seed_generator=GdeltSeedGenerator(
        start_date=datetime(2024, 7, 3),
        end_date=datetime(2025, 11, 30),
        interval_duration_days=7,
        articles_per_interval=25,
    ),
    question_generator=ForwardLookingQuestionGenerator(
        instructions=instructions,
        examples=good_examples,
        bad_examples=bad_examples,
        questions_per_seed=5,
        answer_type=answer_type,
    ),
    context_generators=[NewsContextGenerator(num_articles=10)],
    labeler=WebSearchLabeler(
        answer_type=answer_type,
        confidence_threshold=0.9,
    ),
    renderer=QuestionRenderer(answer_type=answer_type),
)

dataset = lr.transforms.run(pipeline, max_questions=9000)